In [2]:
# EXTRAÇÃO DE ONSETS (TEMPOS EXATOS DOS ESTÍMULOS)

import os
import pandas as pd
import mne
import warnings

# Silenciando avisos para terminal limpo
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

print("\n" + "="*80)
print("Iniciando Mineração de Triggers (FF, FN, FR) no arquivo bruto...")
print("="*80)

# 1. Caminhos e Parâmetros
DIR_RAW = '../data/raw/'
DIR_RESULTS = '../results/'

# Lista de pacientes com falha conhecida (herdada do seu setup)
ARQUIVOS_EXCLUIDOS = ['control_14-sem marcacoes', 'TEA_M_22_1']
ESTIMULOS_ALVO = ['FF', 'FN', 'FR']

# Criar pasta de resultados se não existir
os.makedirs(DIR_RESULTS, exist_ok=True)

# 2. Varredura
arquivos_edf = sorted([f for f in os.listdir(DIR_RAW) if f.lower().endswith('.edf')])
dados_onsets = []
pacientes_processados = 0

for arquivo in arquivos_edf:
    paciente_id = arquivo.replace('.edf', '')
    
    # Pula pacientes excluídos
    if paciente_id in ARQUIVOS_EXCLUIDOS:
        continue
        
    caminho_completo = os.path.join(DIR_RAW, arquivo)
    
    try:
        # preload=False garante que só os metadados/eventos sejam lidos (super rápido)
        raw = mne.io.read_raw_edf(caminho_completo, preload=False, encoding='latin1', verbose=False)
        
        contador_estimulos = 0
        
        # Itera sobre todas as anotações do exame
        for ann in raw.annotations:
            descricao = ann['description'].strip().upper()
            
            # Se a descrição for exatamente FF, FN ou FR, nós extraímos
            if descricao in ESTIMULOS_ALVO:
                dados_onsets.append({
                    'Paciente_ID': paciente_id,
                    'Estimulo': descricao,
                    'Onset_Segundos': ann['onset']
                })
                contador_estimulos += 1
                
        print(f"[{paciente_id}] -> {contador_estimulos} estímulos mapeados.")
        pacientes_processados += 1
        
    except Exception as e:
        print(f"[ERRO] Falha ao processar {paciente_id}: {e}")

# 3. Consolidação e Exportação
df_onsets = pd.DataFrame(dados_onsets)

# Salva o arquivo CSV definitivo
caminho_csv = os.path.join(DIR_RESULTS, 'tabela_tempos_estimulos_faces.csv')
df_onsets.to_csv(caminho_csv, index=False)

print("\n" + "="*80)
print("EXTRAÇÃO CONCLUÍDA COM SUCESSO!")
print(f"Total de pacientes processados: {pacientes_processados}")
print(f"Total de estímulos isolados: {len(df_onsets)}")
print(f"Arquivo salvo em: {caminho_csv}")
print("="*80)

# Exibe as primeiras linhas para conferência visual imediata
display(df_onsets.head(10))


Iniciando Mineração de Triggers (FF, FN, FR) no arquivo bruto...
[TEA_G_26] -> 30 estímulos mapeados.
[TEA_G_28] -> 30 estímulos mapeados.
[TEA_G_30] -> 30 estímulos mapeados.
[TEA_G_40] -> 30 estímulos mapeados.
[TEA_L_16] -> 0 estímulos mapeados.
[TEA_L_17] -> 30 estímulos mapeados.
[TEA_L_24] -> 30 estímulos mapeados.
[TEA_L_25] -> 27 estímulos mapeados.
[TEA_L_27] -> 31 estímulos mapeados.
[TEA_L_29] -> 30 estímulos mapeados.
[TEA_L_32] -> 30 estímulos mapeados.
[TEA_L_46] -> 30 estímulos mapeados.
[TEA_M_15] -> 30 estímulos mapeados.
[TEA_M_18] -> 30 estímulos mapeados.
[TEA_M_19] -> 30 estímulos mapeados.
[TEA_M_20] -> 30 estímulos mapeados.
[TEA_M_21] -> 30 estímulos mapeados.
[TEA_M_22] -> 30 estímulos mapeados.
[control_12] -> 30 estímulos mapeados.
[control_13] -> 30 estímulos mapeados.
[control_35] -> 30 estímulos mapeados.
[control_36] -> 30 estímulos mapeados.
[control_37] -> 30 estímulos mapeados.
[control_38] -> 30 estímulos mapeados.
[control_39] -> 30 estímulos mapead

,Paciente_ID,Estimulo,Onset_Segundos
0,TEA_G_26,FF,1820.0
1,TEA_G_26,FN,1824.0
2,TEA_G_26,FR,1828.0
3,TEA_G_26,FF,1832.0
4,TEA_G_26,FN,1836.0
5,TEA_G_26,FR,1840.0
6,TEA_G_26,FF,1844.0
7,TEA_G_26,FN,1848.0
8,TEA_G_26,FR,1852.0
9,TEA_G_26,FF,1857.0
